In [1]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    Model,
    CodeConfiguration,
    Environment
)
from azure.identity import DefaultAzureCredential
from pathlib import Path
import datetime

In [2]:
# Celda 2 - Conectar
ml_client = MLClient.from_config(credential=DefaultAzureCredential())
print(f"Conectado: {ml_client.workspace_name}")

Found the config file in: /config.json


Conectado: mlw-churn-dev


In [3]:
# Celda 3 - Crear endpoint
endpoint_name = "churn-asandoval-endpoint"

endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="Telco churn prediction endpoint",
    auth_mode="key",
)

ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f" Endpoint creado: {endpoint_name}")

/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.8.1) and child packages mlflow-skinny (3.5.0) are different. This may lead to unexpected behavior. Please install the same version of all MLflow packages.
  mlflow.mismatch._check_version_mismatch()


 Endpoint creado: churn-asandoval-endpoint


In [5]:
# Celda 4 - Crear deployment
current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == 'notebooks' else current_dir

# Crear environment (mismo que antes)
env_final = Environment(
    name="churn-env-final",
    description="Final environment with sklearn 1.3",
    conda_file={
        "channels": ["conda-forge"],
        "dependencies": [
            "python=3.10",
            "pip",
            {
                "pip": [
                    "numpy==1.23.5",
                    "scikit-learn==1.3.0",
                    "pandas==2.0.0",
                    "joblib==1.3.0",
                    "azureml-inference-server-http==1.5.0",
                ]
            }
        ]
    },
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04",
)

env_final = ml_client.environments.create_or_update(env_final)
print(f"Environment: {env_final.name}:{env_final.version}")

# Deployment con modelo v4
deployment = ManagedOnlineDeployment(
    name="churn-deployment-final",  # Nuevo nombre
    endpoint_name=endpoint_name,
    model=ml_client.models.get(name="telco-churn-model", version="5"),
    code_configuration=CodeConfiguration(
        code=str(project_root / "src"),
        scoring_script="score.py"
    ),
    environment=f"{env_final.name}:{env_final.version}",
    instance_type="Standard_DS2_v2",
    instance_count=1,
)

ml_client.online_deployments.begin_create_or_update(deployment).result()
print(f"Deployment creado")

Instance type Standard_DS2_v2 may be too small for compute resources. Minimum recommended compute SKU is Standard_DS3_v2 for general purpose endpoints. Learn more about SKUs here: https://learn.microsoft.com/azure/machine-learning/referencemanaged-online-endpoints-vm-sku-list
Check: endpoint churn-asandoval-endpoint exists


..................................................................................Deployment creado


In [ ]:
#ml_client.online_deployments.begin_delete(
#    name="churn-deployment-final",
#    endpoint_name=endpoint_name
#).result()

In [7]:
# Celda 5 - Asignar 100% tráfico al deployment
endpoint.traffic = {"churn-deployment-final": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f"Tráfico asignado!")

Tráfico asignado!


In [10]:
# Probar request real

import requests
import json

# 1. Obtener URL y key del endpoint
endpoint = ml_client.online_endpoints.get(endpoint_name)
scoring_uri = endpoint.scoring_uri
key = ml_client.online_endpoints.get_keys(endpoint_name).primary_key

print(f"Scoring URI: {scoring_uri}")

# 2. Datos de prueba (un cliente)
data = {
    "customerID": ["CUST-001", "CUST-002", "CUST-003", "CUST-004"],
    "gender": ["Female", "Male", "Female", "Male"],
    "SeniorCitizen": [0, 1, 0, 0],
    "Partner": ["Yes", "No", "Yes", "No"],
    "Dependents": ["No", "No", "Yes", "No"],
    "tenure": [12, 36, 6, 48],
    "PhoneService": ["Yes", "Yes", "Yes", "No"],
    "MultipleLines": ["No", "Yes", "No", "No phone service"],
    "InternetService": ["Fiber optic", "DSL", "Fiber optic", "No"],
    "OnlineSecurity": ["No", "Yes", "No", "No internet service"],
    "OnlineBackup": ["No", "Yes", "No", "No internet service"],
    "DeviceProtection": ["No", "No", "No", "No internet service"],
    "TechSupport": ["No", "Yes", "No", "No internet service"],
    "StreamingTV": ["No", "Yes", "No", "No internet service"],
    "StreamingMovies": ["No", "No", "No", "No internet service"],
    "Contract": ["Month-to-month", "Two year", "Month-to-month", "One year"],
    "PaperlessBilling": ["Yes", "No", "Yes", "No"],
    "PaymentMethod": ["Electronic check", "Bank transfer", "Electronic check", "Mailed check"],
    "MonthlyCharges": [70.0, 55.0, 85.0, 25.0],
    "TotalCharges": [840.0, 1980.0, 510.0, 1200.0]
}

# 3. Hacer request
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {key}"
}

response = requests.post(scoring_uri, headers=headers, data=json.dumps(data))

# 4. Ver resultado
print(f"\nStatus: {response.status_code}")
print(f"Response: {response.json()}")

Scoring URI: https://churn-asandoval-endpoint.eastus2.inference.ml.azure.com/score

Status: 200
Response: {"error": "\"['customerID'] not in index\""}
